# Task 2 — Dataset Exploration
## Fruit & Vegetable Disease Dataset (Healthy vs Rotten)

This notebook explores the raw dataset before any model training.

We examine:
- class distribution
- label structure
- sample images per class
- image size consistency
- potential quality issues such as blur, corrupted files, and suspicious samples

**Expected dataset root:** `../data/raw/`

In [ ]:
import os
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile, UnidentifiedImageError, ImageFilter
from IPython.display import display, Markdown

ImageFile.LOAD_TRUNCATED_IMAGES = True
random.seed(42)
np.random.seed(42)

DATA_ROOT = Path("../data/raw")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Dataset root:", DATA_ROOT.resolve())
print("Exists:", DATA_ROOT.exists())

## 1. Load Class Structure

This cell discovers leaf folders that contain images and turns them into class labels.
It supports either:
- direct class folders inside `data/raw/`, or
- nested folders such as `Produce/Healthy` and `Produce/Rotten`

In [ ]:
def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTS

def collect_leaf_image_dirs(root: Path):
    leaf_dirs = []
    for current_root, _, files in os.walk(root):
        current_path = Path(current_root)
        image_files = [current_path / f for f in files if Path(f).suffix.lower() in IMAGE_EXTS]
        if image_files:
            leaf_dirs.append((current_path, sorted(image_files)))
    return leaf_dirs

def normalise_class_name(leaf_dir: Path, root: Path) -> str:
    rel_parts = leaf_dir.relative_to(root).parts
    if len(rel_parts) >= 2 and rel_parts[-1].lower() in {"healthy", "rotten"}:
        produce = "_".join(rel_parts[:-1]).replace(" ", "_")
        label = rel_parts[-1].capitalize()
        return f"{produce}__{label}"
    return "_".join(rel_parts).replace(" ", "_")

leaf_dirs = collect_leaf_image_dirs(DATA_ROOT)
class_info = {}

for leaf_dir, image_files in leaf_dirs:
    class_name = normalise_class_name(leaf_dir, DATA_ROOT)
    class_info[class_name] = image_files

if not class_info:
    raise FileNotFoundError(
        f"No image classes were found under {DATA_ROOT.resolve()}. "
        "Check the path and dataset structure."
    )

class_counts = {cls: len(paths) for cls, paths in class_info.items()}
class_df = (
    pd.DataFrame({"class_name": list(class_counts.keys()), "count": list(class_counts.values())})
    .sort_values("class_name")
    .reset_index(drop=True)
)

display(class_df)
print(f"Number of classes: {len(class_info)}")
print(f"Total images: {sum(class_counts.values())}")

## Findings — Label Structure

Use the table above to confirm:
- how many classes exist
- whether the labels follow a consistent naming pattern
- whether the dataset is organised as `produce__Healthy` and `produce__Rotten` pairs
- whether any folder names look inconsistent or incorrectly formatted

## 2. Class Distribution

In [ ]:
plot_df = class_df.sort_values("count", ascending=False).reset_index(drop=True)
colors = ["#4CAF50" if "Healthy" in cls else "#E53935" for cls in plot_df["class_name"]]

plt.figure(figsize=(16, 6))
bars = plt.bar(plot_df["class_name"], plot_df["count"], color=colors)

for bar, value in zip(bars, plot_df["count"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(plot_df["count"]) * 0.01,
        str(value),
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of images")
plt.title("Class distribution")
plt.tight_layout()
plt.show()

display(plot_df)
print("Mean images per class:", round(plot_df["count"].mean(), 2))
print("Min class size:", int(plot_df["count"].min()))
print("Max class size:", int(plot_df["count"].max()))

## Findings — Class Distribution

Comment here on:
- whether the dataset is balanced or imbalanced across classes
- which classes have the highest and lowest number of images
- whether any classes are so small that they may cause training bias later

## 3. Sample Images per Class

In [ ]:
def show_sample_images_per_class(class_info, samples_per_class=5, max_classes=None):
    items = list(class_info.items())
    if max_classes is not None:
        items = items[:max_classes]

    for class_name, image_paths in items:
        chosen = image_paths if len(image_paths) <= samples_per_class else random.sample(image_paths, samples_per_class)

        fig, axes = plt.subplots(1, len(chosen), figsize=(3 * len(chosen), 3))
        if len(chosen) == 1:
            axes = [axes]

        for ax, img_path in zip(axes, chosen):
            try:
                img = Image.open(img_path).convert("RGB")
                ax.imshow(img)
                ax.set_title(img_path.name, fontsize=8)
                ax.axis("off")
            except Exception as e:
                ax.text(0.5, 0.5, f"Error\n{type(e).__name__}", ha="center", va="center")
                ax.axis("off")

        fig.suptitle(class_name, fontsize=14)
        plt.tight_layout()
        plt.show()

show_sample_images_per_class(class_info, samples_per_class=5)

## Findings — Visual Inspection

Comment here on:
- whether the images look correctly labelled
- whether there are obvious background differences between classes
- whether some images appear blurrier, darker, cropped differently, or lower quality
- whether there are duplicate-looking or suspicious samples that should be checked manually

## 4. Image Size and Basic Quality Checks

In [ ]:
def sharpness_score(img: Image.Image) -> float:
    gray = img.convert("L")
    edges = gray.filter(ImageFilter.FIND_EDGES)
    return float(np.asarray(edges, dtype=np.float32).var())

records = []
corrupted_files = []

for class_name, image_paths in class_info.items():
    for img_path in image_paths:
        try:
            with Image.open(img_path) as img:
                width, height = img.size
                channels = len(img.getbands())
                score = sharpness_score(img)
                records.append({
                    "class_name": class_name,
                    "path": str(img_path),
                    "filename": img_path.name,
                    "width": width,
                    "height": height,
                    "channels": channels,
                    "aspect_ratio": round(width / height, 4) if height else np.nan,
                    "sharpness": score,
                })
        except (UnidentifiedImageError, OSError, ValueError) as e:
            corrupted_files.append((str(img_path), type(e).__name__))

quality_df = pd.DataFrame(records)

display(quality_df.head())

print("Readable images:", len(quality_df))
print("Corrupted / unreadable images:", len(corrupted_files))
print("Unique image sizes:", quality_df[["width", "height"]].drop_duplicates().shape[0])
print("Unique channel counts:", sorted(quality_df["channels"].dropna().unique().tolist()))

size_counts = (
    quality_df.groupby(["width", "height"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(size_counts.head(10))

In [ ]:
plt.figure(figsize=(10, 5))
quality_df["sharpness"].hist(bins=40)
plt.xlabel("Sharpness score")
plt.ylabel("Number of images")
plt.title("Image sharpness distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
size_summary = quality_df.groupby(quality_df["width"].astype(str) + "x" + quality_df["height"].astype(str)).size().sort_values(ascending=False).head(15)
size_summary.plot(kind="bar")
plt.ylabel("Number of images")
plt.title("Top image sizes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
median_sharpness = quality_df["sharpness"].median()
q1 = quality_df["sharpness"].quantile(0.25)
q3 = quality_df["sharpness"].quantile(0.75)
iqr = q3 - q1
blur_threshold = max(0.0, q1 - 1.5 * iqr)

most_common_size = (
    quality_df.groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
    .index[0]
)

suspicious_df = quality_df[
    (quality_df["sharpness"] <= blur_threshold) |
    ((quality_df["width"] != most_common_size[0]) | (quality_df["height"] != most_common_size[1]))
].copy()

suspicious_df = suspicious_df.sort_values(["sharpness", "width", "height"]).reset_index(drop=True)

print("Blur threshold:", round(blur_threshold, 4))
print("Most common image size:", most_common_size)
print("Suspicious samples flagged:", len(suspicious_df))

display(suspicious_df.head(20))

In [ ]:
def show_suspicious_samples(df, n=12):
    sample_df = df.head(n)
    if sample_df.empty:
        print("No suspicious samples were flagged by the current heuristic.")
        return

    cols = 4
    rows = int(np.ceil(len(sample_df) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sample_df.iterrows()):
        try:
            img = Image.open(row["path"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(
                f'{row["class_name"]}\n{row["width"]}x{row["height"]}\nsharp={row["sharpness"]:.1f}',
                fontsize=8
            )
            ax.axis("off")
        except Exception as e:
            ax.text(0.5, 0.5, f"Error\n{type(e).__name__}", ha="center", va="center")
            ax.axis("off")

    for ax in axes[len(sample_df):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_suspicious_samples(suspicious_df, n=12)

## Findings — Image Quality

Comment here on:
- whether image sizes are consistent or mixed
- whether blurred or low-detail images appear in the dataset
- whether any unreadable or corrupted files were found
- whether the suspicious samples look genuinely problematic or are just valid edge cases
- whether any samples appear mislabelled and should be removed or relabelled manually

## 5. Summary

Write a short conclusion here covering:
- the number of classes and total images
- whether the class distribution is balanced
- whether label naming is clean and consistent
- whether image quality looks acceptable for training
- what preprocessing or cleaning steps should be done next